# LESSON 5.5: Filtered Back Projection (FBP)
## Image Reconstruction from Projections

In this lesson:
- The Filtered Back Projection algorithm
- Step-by-step FBP implementation
- The ramp (Ram-Lak) filter
- Complete FBP reconstruction pipeline
- Comparison with unfiltered back projection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.transform import radon, iradon
from skimage.data import shepp_logan_phantom
from skimage.transform import rescale
from skimage.draw import disk, ellipse

## 1. The FBP Algorithm

**Filtered Back Projection** reconstructs an image from its projections in two steps:

### Step 1: Filter each projection

$$\tilde{g}(\rho, \theta) = \mathcal{F}^{-1}\{ G(\omega, \theta) \cdot |\omega| \}$$

Where:
- $G(\omega, \theta)$ = 1-D FT of the projection $g(\rho, \theta)$
- $|\omega|$ = the ramp filter (compensates for density imbalance)
- $\tilde{g}$ = filtered projection

### Step 2: Back project the filtered projections

$$\boxed{f(x, y) = \int_0^{\pi} \tilde{g}(x\cos\theta + y\sin\theta, \theta) \, d\theta}$$

### Algorithm Summary:
```
For each projection angle θ:
    1. Take the projection g(ρ, θ)
    2. Compute 1-D FFT: G(ω, θ) = FFT{g(ρ, θ)}
    3. Multiply by ramp filter: G̃(ω, θ) = G(ω, θ) × |ω|
    4. Compute inverse FFT: g̃(ρ, θ) = IFFT{G̃(ω, θ)}
    5. Back project g̃(ρ, θ) and add to reconstruction
```

In [ ]:
# Visualize the FBP pipeline for one projection
phantom = shepp_logan_phantom()
phantom = rescale(phantom, 0.5, anti_aliasing=True)

theta = np.array([45.0])  # single angle
projection = radon(phantom, theta=theta, circle=True)
proj = projection[:, 0]

# Step 1: FFT of projection
n = len(proj)
proj_fft = np.fft.fft(proj)
freqs = np.fft.fftfreq(n)

# Step 2: Apply ramp filter
ramp_filter = 2 * np.abs(freqs)
proj_filtered_fft = proj_fft * ramp_filter

# Step 3: Inverse FFT
proj_filtered = np.real(np.fft.ifft(proj_filtered_fft))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Row 1: Spatial domain
axes[0, 0].plot(proj, 'b-', linewidth=2)
axes[0, 0].set_title('1. Original Projection g(ρ, 45°)', fontsize=11)
axes[0, 0].set_xlabel('ρ')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(np.fft.fftshift(freqs), np.fft.fftshift(ramp_filter), 'r-', linewidth=2)
axes[0, 1].set_title('2. Ramp Filter |ω|', fontsize=11)
axes[0, 1].set_xlabel('Frequency ω')
axes[0, 1].grid(True, alpha=0.3)

axes[0, 2].plot(proj_filtered, 'g-', linewidth=2)
axes[0, 2].set_title('3. Filtered Projection g̃(ρ, 45°)', fontsize=11)
axes[0, 2].set_xlabel('ρ')
axes[0, 2].grid(True, alpha=0.3)

# Row 2: Frequency domain
axes[1, 0].plot(np.fft.fftshift(freqs), np.fft.fftshift(np.abs(proj_fft)), 'b-', linewidth=2)
axes[1, 0].set_title('|G(ω, 45°)| (spectrum)', fontsize=11)
axes[1, 0].set_xlabel('Frequency ω')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(np.fft.fftshift(freqs), np.fft.fftshift(ramp_filter), 'r-', linewidth=2)
axes[1, 1].set_title('|ω| (ramp filter)', fontsize=11)
axes[1, 1].set_xlabel('Frequency ω')
axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].plot(np.fft.fftshift(freqs), np.fft.fftshift(np.abs(proj_filtered_fft)), 'g-', linewidth=2)
axes[1, 2].set_title('|G(ω, 45°) × |ω|| (filtered)', fontsize=11)
axes[1, 2].set_xlabel('Frequency ω')
axes[1, 2].grid(True, alpha=0.3)

plt.suptitle('FBP Pipeline: Filter Each Projection Before Back Projecting',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("The ramp filter amplifies high frequencies (edges) and suppresses")
print("the low-frequency bias from the 1/r blurring.")
print("\nNote the negative lobes in the filtered projection — these cancel")
print("out the smearing from neighboring rays during back projection.")

## 2. Implementing FBP from Scratch

Let's implement the complete FBP algorithm step by step.

In [ ]:
def fbp_reconstruct(sinogram, theta, filter_name='ramp'):
    """Filtered Back Projection reconstruction from scratch."""
    n_detector, n_angles = sinogram.shape
    
    # Pad to next power of 2 for efficient FFT
    n_padded = int(2 ** np.ceil(np.log2(2 * n_detector)))
    
    # Create the ramp filter
    freqs = np.fft.fftfreq(n_padded)
    if filter_name == 'ramp':
        filt = 2 * np.abs(freqs)
    elif filter_name == 'shepp-logan':
        filt = 2 * np.abs(freqs) * np.sinc(freqs / 1.0)
    elif filter_name == 'cosine':
        filt = 2 * np.abs(freqs) * np.cos(np.pi * freqs)
    elif filter_name == 'hamming':
        filt = 2 * np.abs(freqs) * (0.54 + 0.46 * np.cos(2 * np.pi * freqs))
    else:
        filt = np.ones(n_padded)  # no filter
    
    # Filter each projection
    filtered_sinogram = np.zeros_like(sinogram)
    for i in range(n_angles):
        # Pad the projection
        proj_padded = np.zeros(n_padded)
        proj_padded[:n_detector] = sinogram[:, i]
        
        # FFT → multiply by filter → IFFT
        proj_fft = np.fft.fft(proj_padded)
        proj_filtered = np.real(np.fft.ifft(proj_fft * filt))
        filtered_sinogram[:, i] = proj_filtered[:n_detector]
    
    # Back projection
    output_size = n_detector
    reconstruction = np.zeros((output_size, output_size))
    center = output_size // 2
    
    # Detector positions (centered)
    detector_pos = np.arange(n_detector) - n_detector // 2
    
    y_grid, x_grid = np.mgrid[:output_size, :output_size] - center
    
    for i in range(n_angles):
        angle_rad = np.radians(theta[i])
        # For each pixel, find which detector it projects to
        rho = x_grid * np.cos(angle_rad) + y_grid * np.sin(angle_rad)
        
        # Interpolate projection value
        rho_idx = rho + n_detector // 2
        rho_idx = np.clip(rho_idx, 0, n_detector - 2)
        
        # Linear interpolation
        idx_low = rho_idx.astype(int)
        idx_high = idx_low + 1
        idx_high = np.clip(idx_high, 0, n_detector - 1)
        frac = rho_idx - idx_low
        
        reconstruction += (1 - frac) * filtered_sinogram[idx_low, i] + frac * filtered_sinogram[idx_high, i]
    
    reconstruction *= np.pi / (2 * n_angles)
    return reconstruction

# Test our implementation
phantom = shepp_logan_phantom()
phantom = rescale(phantom, 0.5, anti_aliasing=True)

theta = np.linspace(0., 180., 180, endpoint=False)
sinogram = radon(phantom, theta=theta, circle=True)

# Our implementation
recon_ours = fbp_reconstruct(sinogram, theta, filter_name='ramp')

# scikit-image implementation
recon_skimage = iradon(sinogram, theta=theta, filter_name='ramp', circle=True)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(phantom, cmap='gray')
axes[0].set_title('Original', fontsize=12)

axes[1].imshow(sinogram, cmap='hot', aspect='auto',
              extent=[0, 180, sinogram.shape[0], 0])
axes[1].set_title('Sinogram', fontsize=12)

axes[2].imshow(recon_ours, cmap='gray')
axes[2].set_title('Our FBP', fontsize=12)

axes[3].imshow(recon_skimage, cmap='gray')
axes[3].set_title('scikit-image FBP', fontsize=12)

plt.suptitle('FBP Implementation: From Scratch vs scikit-image',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Our FBP implementation produces results comparable to scikit-image's iradon().")

## 3. Filtered vs Unfiltered: Visual Comparison

Let's directly compare unfiltered and filtered back projection at each step.

In [ ]:
# Side-by-side comparison at different stages
phantom = shepp_logan_phantom()
phantom = rescale(phantom, 0.5, anti_aliasing=True)

stages = [5, 15, 30, 60, 90, 180]

fig, axes = plt.subplots(3, len(stages), figsize=(20, 10))

all_angles = np.linspace(0., 180., 180, endpoint=False)
sinogram_full = radon(phantom, theta=all_angles, circle=True)

for i, n in enumerate(stages):
    theta = all_angles[:n]
    sinogram = sinogram_full[:, :n]
    
    # Sinogram
    axes[0, i].imshow(sinogram, cmap='hot', aspect='auto')
    axes[0, i].set_title(f'{n} projections', fontsize=10)
    if i == 0:
        axes[0, i].set_ylabel('Sinogram', fontsize=11)
    
    # Unfiltered BP
    recon_uf = iradon(sinogram, theta=theta, filter_name=None, circle=True)
    axes[1, i].imshow(recon_uf, cmap='gray')
    if i == 0:
        axes[1, i].set_ylabel('Unfiltered BP', fontsize=11)
    
    # Filtered BP
    recon_f = iradon(sinogram, theta=theta, filter_name='ramp', circle=True)
    axes[2, i].imshow(recon_f, cmap='gray')
    if i == 0:
        axes[2, i].set_ylabel('Filtered BP', fontsize=11)

plt.suptitle('Unfiltered vs Filtered Back Projection: Progressive Reconstruction',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Row 1: Sinograms with increasing number of projections")
print("Row 2: Unfiltered BP — always blurry regardless of projection count")
print("Row 3: Filtered BP — sharp reconstruction that improves with more projections")

## 4. Effect of the Ramp Filter on Projections

Let's visualize what the ramp filter does to the sinogram.

In [ ]:
# Unfiltered vs filtered sinograms
phantom = shepp_logan_phantom()
phantom = rescale(phantom, 0.5, anti_aliasing=True)

theta = np.linspace(0., 180., 180, endpoint=False)
sinogram = radon(phantom, theta=theta, circle=True)

# Apply ramp filter to each projection
n_det = sinogram.shape[0]
freqs = np.fft.fftfreq(n_det)
ramp = 2 * np.abs(freqs)

sinogram_filtered = np.zeros_like(sinogram)
for i in range(sinogram.shape[1]):
    proj_fft = np.fft.fft(sinogram[:, i])
    sinogram_filtered[:, i] = np.real(np.fft.ifft(proj_fft * ramp))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Sinograms
axes[0, 0].imshow(sinogram, cmap='hot', aspect='auto',
                  extent=[0, 180, n_det, 0])
axes[0, 0].set_title('Original Sinogram', fontsize=12)
axes[0, 0].set_xlabel('θ (degrees)')
axes[0, 0].set_ylabel('ρ')

axes[0, 1].imshow(sinogram_filtered, cmap='seismic', aspect='auto',
                  extent=[0, 180, n_det, 0])
axes[0, 1].set_title('Filtered Sinogram (ramp)', fontsize=12)
axes[0, 1].set_xlabel('θ (degrees)')

# Difference
axes[0, 2].imshow(sinogram - sinogram_filtered, cmap='seismic', aspect='auto',
                  extent=[0, 180, n_det, 0])
axes[0, 2].set_title('Difference', fontsize=12)
axes[0, 2].set_xlabel('θ (degrees)')

# Individual projection comparison
angle_idx = 45  # projection at ~45 degrees
axes[1, 0].plot(sinogram[:, angle_idx], 'b-', linewidth=2)
axes[1, 0].set_title('Original Projection at 45°', fontsize=11)
axes[1, 0].set_xlabel('ρ')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(sinogram_filtered[:, angle_idx], 'r-', linewidth=2)
axes[1, 1].set_title('Filtered Projection at 45°', fontsize=11)
axes[1, 1].set_xlabel('ρ')
axes[1, 1].grid(True, alpha=0.3)

# Overlay
axes[1, 2].plot(sinogram[:, angle_idx] / sinogram[:, angle_idx].max(),
               'b-', linewidth=2, label='Original', alpha=0.7)
axes[1, 2].plot(sinogram_filtered[:, angle_idx] / np.abs(sinogram_filtered[:, angle_idx]).max(),
               'r-', linewidth=2, label='Filtered', alpha=0.7)
axes[1, 2].set_title('Overlay (normalized)', fontsize=11)
axes[1, 2].set_xlabel('ρ')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.suptitle('Ramp Filtering: Before and After', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("After filtering: edges are enhanced and negative lobes appear.")
print("The negative lobes cancel out the smearing during back projection.")

## 5. Quality Assessment

Let's quantitatively assess the FBP reconstruction quality.

In [ ]:
# Quality assessment of FBP reconstruction
phantom = shepp_logan_phantom()
phantom = rescale(phantom, 0.5, anti_aliasing=True)

theta = np.linspace(0., 180., 180, endpoint=False)
sinogram = radon(phantom, theta=theta, circle=True)

# Reconstruct
recon = iradon(sinogram, theta=theta, filter_name='ramp', circle=True)

# Crop to same size for comparison
min_dim = min(phantom.shape[0], recon.shape[0])
phantom_crop = phantom[:min_dim, :min_dim]
recon_crop = recon[:min_dim, :min_dim]

# Error image
error = phantom_crop - recon_crop

# Metrics
rmse = np.sqrt(np.mean(error**2))
psnr = 20 * np.log10(phantom_crop.max() / rmse) if rmse > 0 else float('inf')

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

axes[0, 0].imshow(phantom_crop, cmap='gray')
axes[0, 0].set_title('Original', fontsize=12)

axes[0, 1].imshow(recon_crop, cmap='gray')
axes[0, 1].set_title('FBP Reconstruction', fontsize=12)

im_err = axes[0, 2].imshow(error, cmap='seismic', vmin=-0.1, vmax=0.1)
axes[0, 2].set_title(f'Error (RMSE={rmse:.4f}, PSNR={psnr:.1f}dB)', fontsize=11)
plt.colorbar(im_err, ax=axes[0, 2])

# Line profiles
center = min_dim // 2
# Horizontal profile
axes[1, 0].plot(phantom_crop[center, :], 'b-', linewidth=2, label='Original')
axes[1, 0].plot(recon_crop[center, :], 'r--', linewidth=2, label='FBP')
axes[1, 0].set_title('Horizontal Profile (center row)', fontsize=11)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Vertical profile
axes[1, 1].plot(phantom_crop[:, center], 'b-', linewidth=2, label='Original')
axes[1, 1].plot(recon_crop[:, center], 'r--', linewidth=2, label='FBP')
axes[1, 1].set_title('Vertical Profile (center col)', fontsize=11)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Error histogram
axes[1, 2].hist(error.ravel(), bins=100, color='steelblue', alpha=0.7)
axes[1, 2].set_title('Error Distribution', fontsize=11)
axes[1, 2].set_xlabel('Error value')
axes[1, 2].set_ylabel('Count')
axes[1, 2].axvline(x=0, color='r', linestyle='--')
axes[1, 2].grid(True, alpha=0.3)

plt.suptitle('FBP Reconstruction Quality Assessment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"RMSE: {rmse:.6f}")
print(f"PSNR: {psnr:.2f} dB")
print(f"Max absolute error: {np.abs(error).max():.6f}")

## 6. FBP with Different Numbers of Projections

How many projections do we need for a good reconstruction?

In [ ]:
# FBP quality vs number of projections
phantom = shepp_logan_phantom()
phantom = rescale(phantom, 0.5, anti_aliasing=True)

n_proj_list = [10, 20, 45, 90, 180, 360]

fig, axes = plt.subplots(2, len(n_proj_list), figsize=(20, 7))

for i, n_proj in enumerate(n_proj_list):
    theta = np.linspace(0., 180., n_proj, endpoint=False)
    sinogram = radon(phantom, theta=theta, circle=True)
    recon = iradon(sinogram, theta=theta, filter_name='ramp', circle=True)
    
    axes[0, i].imshow(recon, cmap='gray')
    axes[0, i].set_title(f'{n_proj} projections', fontsize=10)
    axes[0, i].axis('off')
    
    # Error image
    min_dim = min(phantom.shape[0], recon.shape[0])
    error = phantom[:min_dim, :min_dim] - recon[:min_dim, :min_dim]
    rmse = np.sqrt(np.mean(error**2))
    
    axes[1, i].imshow(np.abs(error), cmap='hot', vmin=0, vmax=0.3)
    axes[1, i].set_title(f'|Error| (RMSE={rmse:.3f})', fontsize=9)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Reconstruction', fontsize=11)
axes[1, 0].set_ylabel('Absolute Error', fontsize=11)

plt.suptitle('FBP Reconstruction Quality vs Number of Projections',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("10 projections: Strong streak artifacts")
print("45 projections: Artifacts visible but structure clear")
print("180 projections: Good quality reconstruction")
print("360 projections: Excellent reconstruction")

## Summary

What we learned:
1. **Filtered Back Projection** (FBP) is a two-step algorithm: **filter** each projection, then **back project**
2. The **ramp filter** $|\omega|$ compensates for the $1/r$ blurring of unfiltered back projection
3. The ramp filter enhances edges and creates **negative lobes** that cancel smearing
4. FBP produces **dramatically sharper** results than unfiltered back projection
5. More projections → fewer streak artifacts → better reconstruction quality
6. FBP is the **standard algorithm** used in clinical CT scanners
7. The pure ramp filter amplifies noise — we need **windowed filters** (next lesson)